# Closed-Loop Financial Risk MLOps Platform: GPU Acceleration & AgentCore LLM on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devdasrahul/careerhive-agent/blob/main/notebooks/colab_gpu_fraud_pipeline.ipynb)

This notebook demonstrates where **GPU Acceleration** is utilized in the **Financial Risk MLOps Platform**:
1. **Large-Scale Tabular Training (XGBoost on CUDA GPU):** Training on hundreds of thousands of transactions with `tree_method='hist', device='cuda'`.
2. **Agentic RAG Compliance Triage with Open-Source LLMs:** Serving the **Bedrock AgentCore** compliance triage agent using an open-source instruct model (e.g. `Qwen/Qwen2.5-1.5B-Instruct` or `google/gemma-2-2b-it`) running on free Google Colab Tesla T4 GPU.

## 1. Verify NVIDIA GPU Environment
Ensure your Google Colab runtime is set to **T4 GPU** (**Runtime** -> **Change runtime type** -> **T4 GPU**).

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Install Dependencies
Install XGBoost, SHAP, MLflow, and Hugging Face Transformers.

In [ ]:
!pip install -q xgboost>=2.0.3 shap>=0.45.0 mlflow>=2.11.0 transformers>=4.38.0 accelerate>=0.27.0 pyarrow pandas numpy scikit-learn

## 3. Generate Large-Scale Scaled Transaction Dataset (100,000 Samples)
Simulates high-velocity credit card transactions with realistic imbalanced fraud rates (~2.5%).

In [ ]:
import numpy as np
import pandas as pd
import time

np.random.seed(42)
n_samples = 100_000
print(f"Generating {n_samples:,} transaction records...")

is_international = (np.random.rand(n_samples) < 0.08).astype(int)
velocity_1h = np.random.poisson(lam=0.4, size=n_samples)
velocity_24h = velocity_1h + np.random.poisson(lam=1.8, size=n_samples)
distance_km = np.random.exponential(scale=20.0, size=n_samples)
merchant_risk = np.clip(np.random.beta(2, 8, size=n_samples), 0.01, 0.99)
device_trust = np.clip(np.random.beta(8, 2, size=n_samples), 0.01, 1.0)
account_age_days = np.random.exponential(scale=500, size=n_samples).astype(int) + 30
amount = np.exp(np.random.normal(3.8, 0.9, size=n_samples))

# Ground truth fraud
is_fraud = (
    (np.random.rand(n_samples) < 0.02) |
    ((is_international == 1) & (merchant_risk > 0.7) & (velocity_1h >= 3))
).astype(int)

df = pd.DataFrame({
    'transaction_amount': amount,
    'card_velocity_1h': velocity_1h,
    'card_velocity_24h': velocity_24h,
    'distance_from_home_km': distance_km,
    'merchant_risk_score': merchant_risk,
    'is_international': is_international,
    'device_trust_score': device_trust,
    'account_age_days': account_age_days,
    'amount_per_velocity_1h': amount / (velocity_1h + 1.0),
    'velocity_ratio_1h_to_24h': (velocity_1h + 0.1) / (velocity_24h + 1.0),
    'composite_risk_indicator': merchant_risk * (1.0 - device_trust),
    'is_fraud': is_fraud
})

print(f"Dataset shape: {df.shape} | Fraud percentage: {df['is_fraud'].mean()*100:.2f}%")

## 4. Benchmark: CPU vs GPU Training Time with XGBoost
Compare the execution time of CPU vs CUDA GPU on 100k samples.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, average_precision_score

features = [c for c in df.columns if c != 'is_fraud']
X_train, X_val, y_train, y_val = train_test_split(df[features], df['is_fraud'], test_size=0.2, random_state=42)

# 1. GPU Training with CUDA
print("=== Training with GPU (device='cuda') ===")
t0 = time.time()
model_gpu = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.08,
    scale_pos_weight=35.0,
    tree_method='hist',
    device='cuda',          # Enables GPU
    eval_metric='aucpr',
    random_state=42
)
model_gpu.fit(X_train, y_train)
gpu_duration = time.time() - t0
print(f"GPU Training Time: {gpu_duration:.2f} seconds")

val_probs = model_gpu.predict_proba(X_val)[:, 1]
print(f"GPU Model PR-AUC: {average_precision_score(y_val, val_probs):.4f}")

# Save trained model for local platform use
model_gpu.save_model("gpu_fraud_model.json")
print("Saved GPU model artifact to gpu_fraud_model.json")

## 5. AgentCore Compliance Triage Agent: Running Open-Source LLM on Free T4 GPU
Demonstrates the **AgentCore Triage Agent** running a real open-source Large Language Model (`Qwen/Qwen2.5-1.5B-Instruct` or `google/gemma-2-2b-it`) on GPU to generate regulatory compliance memos.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {model_id} onto GPU...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
llm = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded on GPU successfully!")

### AgentCore Prompt Formulation with SHAP Values & AML/KYC Policy Rules

In [ ]:
system_prompt = """
You are an AI Compliance Investigator operating under Amazon Bedrock AgentCore governance.
Review flagged high-risk payment transactions, apply FinCEN / Bank Secrecy Act (BSA) rules, and output a structured compliance audit memo.
"""

transaction_context = """
Flagged Transaction ID: TX_20261015_0099412
Fraud Probability: 99.8%
Risk Tier: CRITICAL
Masked Card: 411111******8821
Top SHAP Attributions:
- card_velocity_1h = 5.0 (SHAP: +6.82, High burst structuring)
- card_velocity_24h = 14.0 (SHAP: +2.15, Elevated aggregate movement)
- merchant_risk_score = 0.94 (SHAP: +1.42, Unverified crypto on-ramp)
- is_international = 1 (SHAP: +0.89, Foreign terminal location)

Applicable Policy: FinCEN High-Velocity Structuring Rule (31 CFR §1010.311)
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Generate an auditable compliance memo with: Recommended Action, Regulatory Policy Citation, and Narrative Rationale based on:\n{transaction_context}"}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to("cuda")

t0 = time.time()
outputs = llm.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.2,
    do_sample=True
)
duration = time.time() - t0

response_text = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
print(f"\n=== AGENTCORE GPU COMPLIANCE MEMO (Generated in {duration:.2f}s) ===\n")
print(response_text)